# yeast4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('42_yeast4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 0


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,0.000,0.000,0.000,0.000,100.000,50.000,47.221,85.361,0.000
smote,71.547,31.758,22.457,56.909,93.090,75.000,41.804,90.046,51.653
bsmote,65.410,32.591,25.566,47.455,95.184,71.319,41.798,90.655,49.001
ros,71.799,28.897,19.539,57.091,91.488,74.289,40.633,89.697,50.348
adasyn,70.943,28.456,19.339,56.818,91.519,74.169,40.111,89.332,49.700
casb,70.654,37.176,28.826,54.000,95.256,74.628,44.633,90.516,53.915
casb_relaxed,72.002,36.290,27.156,56.000,94.558,75.279,44.066,90.284,54.146
casb_edited,73.258,32.717,22.890,59.000,92.917,75.959,43.272,90.106,52.988
casb_ros,71.058,34.214,25.354,55.000,94.209,74.604,43.847,90.720,52.636


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,43.627,30.171,78.667,19.636,99.721,59.679,43.705,81.100,36.899
smote,75.097,31.804,21.216,64.727,91.766,78.246,30.658,82.391,53.450
bsmote,72.692,36.570,26.792,59.091,94.383,76.737,32.838,82.150,54.631
ros,72.423,34.684,24.767,59.091,93.790,76.441,27.687,80.550,53.554
adasyn,74.687,29.479,19.200,64.727,90.789,77.758,30.903,82.101,52.083
casb,73.554,35.204,24.950,61.091,93.616,77.353,30.686,81.316,54.379
casb_relaxed,74.976,33.883,23.214,63.818,92.673,78.246,33.189,82.199,54.429
casb_edited,76.502,30.186,19.557,66.727,90.474,78.601,27.566,82.115,53.344
casb_ros,72.437,34.813,24.910,59.091,93.825,76.458,28.022,80.527,53.625


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,20.241,12.902,36.000,8.909,99.721,54.315,44.635,93.283,16.571
smote,67.580,39.225,33.696,49.091,96.615,72.853,41.367,92.179,53.402
bsmote,55.438,32.289,34.342,34.727,97.663,66.195,39.899,92.694,43.863
ros,51.999,34.130,43.386,30.545,98.464,64.505,42.845,93.697,43.064
adasyn,65.498,35.812,30.830,46.364,96.267,71.315,40.820,92.259,50.655
casb,60.178,40.978,48.346,39.455,98.254,68.854,42.428,93.007,50.578
casb_relaxed,64.106,38.483,35.188,44.364,97.104,70.734,42.733,92.753,51.295
casb_edited,73.142,36.626,27.284,57.909,94.487,76.198,42.947,92.421,54.884
casb_ros,45.232,29.921,42.917,25.636,98.604,62.120,40.549,93.247,37.577


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,33.890,19.273,78.000,11.818,99.721,55.769,42.199,87.700,26.581
smote,78.993,31.084,20.078,69.455,89.988,79.721,38.978,87.232,55.039
bsmote,80.925,36.781,24.883,71.273,92.253,81.763,42.424,87.465,58.853
ros,80.652,31.103,19.966,73.364,88.907,81.135,35.958,87.482,55.877
adasyn,79.214,30.299,19.418,70.455,89.186,79.820,37.662,87.238,54.757
casb,79.979,34.316,22.938,70.364,91.173,80.769,39.784,87.387,57.148
casb_relaxed,79.272,33.211,22.024,69.364,90.894,80.129,39.877,87.412,56.242
casb_edited,79.585,30.056,19.145,71.364,88.907,80.135,38.476,87.532,54.821
casb_ros,80.010,34.523,23.130,70.364,91.243,80.804,39.314,87.286,57.266


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,24.439,15.586,48.167,10.091,99.790,54.941,44.440,86.861,20.013
smote,73.304,33.468,24.362,60.045,92.865,76.455,38.202,87.962,53.386
bsmote,68.616,34.558,27.896,53.136,94.871,74.004,39.240,88.241,51.587
ros,69.218,32.203,26.915,55.023,93.162,74.093,36.781,87.857,50.711
adasyn,72.586,31.011,22.197,59.591,91.940,75.766,37.374,87.733,51.799
casb,71.091,36.919,31.265,56.227,94.575,75.401,39.383,88.057,54.005
casb_relaxed,72.589,35.467,26.895,58.386,93.807,76.097,39.966,88.162,54.028
casb_edited,75.622,32.396,22.219,63.750,91.696,77.723,38.065,88.043,54.009
casb_ros,67.184,33.368,29.078,52.523,94.470,73.497,37.933,87.945,50.276


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,73.258,ros,71.799,1.459,WIN
1,f1,casb,37.176,bsmote,32.591,4.585,WIN
2,precision,casb,28.826,bsmote,25.566,3.260,WIN
3,recall,casb_edited,59.000,ros,57.091,1.909,WIN


## knn: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,76.502,smote,75.097,1.406,WIN
1,f1,casb,35.204,bsmote,36.570,-1.366,LOSS
2,precision,casb,24.950,bsmote,26.792,-1.841,LOSS
3,recall,casb_edited,66.727,smote,64.727,2.000,WIN


## rf: per-metric family comparison

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,73.142,smote,67.580,5.562,WIN
1,f1,casb,40.978,smote,39.225,1.752,WIN
2,precision,casb,48.346,ros,43.386,4.961,WIN
3,recall,casb_edited,57.909,smote,49.091,8.818,WIN


... (3 outputs left out)

,metric,best_variant,family_percent,baseline,baseline_percent,difference_pp,result
0,gmean,casb_edited,75.622,smote,73.304,2.318,WIN
1,f1,casb,36.919,bsmote,34.558,2.361,WIN
2,precision,casb,31.265,bsmote,27.896,3.370,WIN
3,recall,casb_edited,63.750,smote,60.045,3.705,WIN


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.50,0.000000,0.000000,0.000000,0.000000,1.000000,0.500000,0.441585,0.872537,0.000000,None
1,42,1,svm,smote,0.50,0.713595,0.333333,0.240000,0.545455,0.933566,0.739510,0.350274,0.933566,0.523464,None
2,42,1,svm,bsmote,0.50,0.513950,0.260870,0.250000,0.272727,0.968531,0.620629,0.345516,0.939606,0.387410,None
3,42,1,svm,ros,0.50,0.647750,0.263158,0.185185,0.454545,0.923077,0.688811,0.279898,0.914177,0.455454,None
4,42,1,svm,adasyn,1.00,0.745830,0.259259,0.162791,0.636364,0.874126,0.755245,0.341900,0.897330,0.502544,None
5,42,1,svm,casb,0.50,0.661107,0.370370,0.312500,0.454545,0.961538,0.708042,0.367074,0.941831,0.515739,None
6,42,1,svm,casb_relaxed,0.50,0.659904,0.357143,0.294118,0.454545,0.958042,0.706294,0.381263,0.934520,0.508524,None
7,42,1,svm,casb_edited,0.50,0.651420,0.285714,0.208333,0.454545,0.933566,0.694056,0.362095,0.932295,0.468567,None
8,42,1,svm,casb_ros,0.50,0.659904,0.357143,0.294118,0.454545,0.958042,0.706294,0.352836,0.940559,0.508524,None
9,42,1,knn,original,0.50,0.425655,0.285714,0.666667,0.181818,0.996503,0.589161,0.395623,0.742212,0.355685,None
